# RAG по GitHub-репозиторию

Ниже — ноутбук для:
- клонирования GitHub-репозитория,
- парсинга Python / Markdown / TXT файлов,
- разбиения на чанки,
- индексации в Qdrant,
- поиска по эмбеддингам,
- реранкинга,
- и ответа через RAG.

Версия подготовлена для корректного отображения на GitHub:
- удалены тяжёлые выводы и widget-метаданные,
- сохранена структура ноутбука,
- код оставлен исполняемым.

## 1. Установка зависимостей

In [ ]:
!pip install -q GitPython pydantic
!pip install -q tree-sitter tree-sitter-python
!pip install -q langchain-text-splitters
!pip install -q sentence-transformers qdrant-client openai

## 2. Импорты

In [ ]:
# Стандартная библиотека
import json
import os
import shutil
from datetime import datetime, UTC
from pathlib import Path
from typing import Any, Dict, List

# Внешние зависимости
import git
import openai
import torch
import tree_sitter_python as tspy

from pydantic import BaseModel
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from sentence_transformers import CrossEncoder, SentenceTransformer
from tree_sitter import Language, Parser
from langchain_text_splitters import MarkdownHeaderTextSplitter

## 3. Базовые структуры данных

In [ ]:
class DocumentChunk(BaseModel):
    content: str
    metadata: Dict[str, Any]


class BaseParser:
    def parse(self, file_content: str, file_path: str, repo_url: str) -> List[DocumentChunk]:
        raise NotImplementedError

## 4. Парсеры файлов

Здесь собраны парсеры для:
- обычных текстовых файлов,
- Python-кода,
- Markdown / TXT с разбиением по заголовкам.

In [ ]:
class TextParser(BaseParser):
    def parse(self, file_content: str, file_path: str, repo_url: str) -> List[DocumentChunk]:
        return [
            DocumentChunk(
                content=file_content,
                metadata={
                    "type": "documentation",
                    "file_path": file_path,
                    "extension": Path(file_path).suffix,
                    "source_url": f"{repo_url}/blob/main/{file_path}",
                },
            )
        ]


class PythonParser(BaseParser):
    def parse(self, file_content: str, file_path: str, repo_url: str) -> List[DocumentChunk]:
        chunks = []
        parts = file_content.split("\n\n")

        for i, part in enumerate(parts):
            if part.strip():
                chunks.append(
                    DocumentChunk(
                        content=part.strip(),
                        metadata={
                            "type": "code",
                            "file_path": file_path,
                            "chunk_id": i,
                            "source_url": f"{repo_url}/blob/main/{file_path}",
                        },
                    )
                )
        return chunks

In [ ]:
class TreeSitterPythonParser(BaseParser):
    def __init__(self):
        self.py_language = Language(tspy.language())
        self.parser = Parser(self.py_language)

    def parse(self, file_content: str, file_path: str, repo_url: str) -> List[DocumentChunk]:
        tree = self.parser.parse(bytes(file_content, "utf8"))
        root_node = tree.root_node
        chunks = []

        for child in root_node.children:
            if child.type in {
                "function_definition",
                "class_definition",
                "import_statement",
                "import_from_statement",
            }:
                start_byte = child.start_byte
                end_byte = child.end_byte
                node_content = file_content[start_byte:end_byte]

                name = "module"
                for grand_child in child.children:
                    if grand_child.type == "identifier":
                        name = file_content[grand_child.start_byte:grand_child.end_byte]
                        break

                start_line = child.start_point[0] + 1
                end_line = child.end_point[0] + 1

                chunks.append(
                    DocumentChunk(
                        content=node_content,
                        metadata={
                            "type": "code",
                            "content_type": child.type,
                            "name": name,
                            "file_path": file_path,
                            "lines": f"{start_line}-{end_line}",
                            "source_url": f"{repo_url}/blob/main/{file_path}#L{start_line}-L{end_line}",
                        },
                    )
                )

        if not chunks and file_content.strip():
            chunks.append(
                DocumentChunk(
                    content=file_content,
                    metadata={
                        "type": "code_raw",
                        "file_path": file_path,
                        "source_url": f"{repo_url}/blob/main/{file_path}",
                    },
                )
            )

        return chunks

In [ ]:
class AdvancedTextParser(BaseParser):
    def __init__(self):
        self.headers_to_split_on = [
            ("#", "Header 1"),
            ("##", "Header 2"),
            ("###", "Header 3"),
        ]
        self.splitter = MarkdownHeaderTextSplitter(
            headers_to_split_on=self.headers_to_split_on
        )

    def parse(self, file_content: str, file_path: str, repo_url: str) -> List[DocumentChunk]:
        splits = self.splitter.split_text(file_content)
        chunks = []

        for i, split in enumerate(splits):
            header_context = " > ".join(
                [v for k, v in split.metadata.items() if "Header" in k]
            )

            chunks.append(
                DocumentChunk(
                    content=split.page_content,
                    metadata={
                        "type": "documentation",
                        "file_path": file_path,
                        "section": header_context,
                        "chunk_id": i,
                        "source_url": f"{repo_url}/blob/main/{file_path}",
                    },
                )
            )

        if not chunks:
            chunks.append(
                DocumentChunk(
                    content=file_content,
                    metadata={
                        "type": "text",
                        "file_path": file_path,
                        "source_url": f"{repo_url}/blob/main/{file_path}",
                    },
                )
            )

        return chunks

## 5. Обработка репозитория

Класс `RepoProcessor`:
- клонирует репозиторий,
- проходит по файлам,
- выбирает нужный парсер по расширению,
- возвращает общий список чанков.

In [ ]:
class RepoProcessor:
    def __init__(self, repo_url: str):
        self.repo_url = repo_url.rstrip("/")
        self.local_path = "./temp_repo"
        self.parsers = {
            ".py": TreeSitterPythonParser(),
            ".md": AdvancedTextParser(),
            ".txt": AdvancedTextParser(),
        }

    def _clone(self):
        if os.path.exists(self.local_path):
            shutil.rmtree(self.local_path)
        git.Repo.clone_from(self.repo_url, self.local_path)

    def process(self) -> List[DocumentChunk]:
        self._clone()
        all_chunks = []

        for root, _, files in os.walk(self.local_path):
            if ".git" in root:
                continue

            for file in files:
                ext = Path(file).suffix
                if ext not in self.parsers:
                    continue

                full_path = os.path.join(root, file)
                relative_path = os.path.relpath(full_path, self.local_path)

                try:
                    with open(full_path, "r", encoding="utf-8") as f:
                        content = f.read()
                except UnicodeDecodeError:
                    continue

                chunks = self.parsers[ext].parse(content, relative_path, self.repo_url)
                all_chunks.extend(chunks)

        return all_chunks

## 6. Запуск парсинга репозитория

In [ ]:
repo_url = "https://github.com/artem-tizhin/llm-hallucination-judge-benchmark-ru"

processor = RepoProcessor(repo_url)
results = processor.process()

print(f"Извлечено чанков: {len(results)}")
print(f"Пример метаданных первого чанка: {results[0].metadata}")

for i in range(min(5, len(results))):
    print(f"\n--- Chunk {i} ---")
    print(results[i].content[:800])

## 7. Векторное хранилище на Qdrant

На этом этапе:
- создаём эмбеддинги через `sentence-transformers`,
- сохраняем их в in-memory Qdrant,
- добавляем поиск по запросу.

In [ ]:
class VectorStore:
    def __init__(
        self,
        collection_name: str = "github_repo",
        embedding_model_name: str = "paraphrase-multilingual-MiniLM-L12-v2",
    ):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Использую устройство: {self.device}")

        self.embedding_model_name = embedding_model_name
        self.model = SentenceTransformer(
            embedding_model_name,
            device=self.device,
        )

        self.client = QdrantClient(":memory:")
        self.collection_name = collection_name

        if self.client.collection_exists(collection_name=self.collection_name):
            self.client.delete_collection(collection_name=self.collection_name)

        self.client.create_collection(
            collection_name=self.collection_name,
            vectors_config=VectorParams(size=384, distance=Distance.COSINE),
        )

    def upsert_chunks(self, chunks: List[DocumentChunk]):
        contents = [chunk.content for chunk in chunks]
        print(f"Начинаю векторизацию {len(contents)} чанков...")

        vectors = self.model.encode(
            contents,
            batch_size=32,
            show_progress_bar=True,
            convert_to_numpy=True,
        ).tolist()

        points = []
        for i, (chunk, vector) in enumerate(zip(chunks, vectors)):
            points.append(
                PointStruct(
                    id=i,
                    vector=vector,
                    payload={
                        "content": chunk.content,
                        **chunk.metadata,
                    },
                )
            )

        self.client.upsert(
            collection_name=self.collection_name,
            points=points,
        )
        print(f"Успешно загружено {len(points)} векторов в Qdrant.")

    def search(self, query: str, limit: int = 3):
        query_vector = self.model.encode(query).tolist()

        search_result = self.client.query_points(
            collection_name=self.collection_name,
            query=query_vector,
            limit=limit,
        ).points

        return search_result

## 8. Индексация и тестовый поиск

In [ ]:
chunks = processor.process()

vstore = VectorStore()
vstore.upsert_chunks(chunks)

query = "Можешь в целом рассказать про что проект? Какие модели использовались для сравнения?"
search_results = vstore.search(query)

print("\n--- Результаты поиска ---")
for res in search_results:
    print(f"Score: {res.score:.3f}")
    print(f"File: {res.payload['file_path']}")
    print(f"Content: {res.payload['content'][:200]}...")
    print("-" * 40)

## 9. Реранкер

Добавляем дополнительный этап сортировки найденных чанков через `CrossEncoder`.

In [ ]:
class BaseReranker:
    def rerank(self, query: str, results: List[Any]) -> List[Any]:
        raise NotImplementedError


class BGEReranker(BaseReranker):
    def __init__(self, model_name: str = "BAAI/bge-reranker-base"):
        print(f"Loading Reranker: {model_name}...")
        self.model_name = model_name
        self.model = CrossEncoder(model_name)

    def rerank(self, query: str, results: List[Any], top_n: int = 4) -> List[Any]:
        if not results:
            return []

        pairs = [[query, res.payload["content"]] for res in results]
        scores = self.model.predict(pairs)

        enriched = []
        for res, score in zip(results, scores):
            enriched.append(
                {
                    "score": float(score),
                    "payload": res.payload,
                }
            )

        enriched.sort(key=lambda x: x["score"], reverse=True)
        return enriched[:top_n]

## 10. RAG-система

Здесь связываем:
- retrieval из Qdrant,
- reranking,
- и генерацию ответа через LLM API.

In [ ]:
class RAGSystem:
    def __init__(
        self,
        vector_store: VectorStore,
        api_key: str,
        base_url: str,
        model_name: str = "llama-3-8b-instruct",
        reranker: BaseReranker | None = None,
    ):
        self.vstore = vector_store
        self.client = openai.OpenAI(api_key=api_key, base_url=base_url)
        self.model = model_name
        self.reranker = reranker

    def ask(self, question: str):
        search_limit = 10 if self.reranker else 4
        search_results = self.vstore.search(question, limit=search_limit)

        if self.reranker:
            reranked = self.reranker.rerank(question, search_results, top_n=4)
            final_results = reranked
        else:
            final_results = [
                {"score": float(res.score), "payload": res.payload}
                for res in search_results
            ]

        context_text = ""
        sources = []

        for res in final_results:
            content = res["payload"]["content"]
            path = res["payload"]["file_path"]
            url = res["payload"].get("source_url", "No URL")

            context_text += f"\n---\nFILE: {path}\nCONTENT:\n{content}\n"
            sources.append(f"- {path} ({url})")

        system_prompt = f"""Ты эксперт по анализу репозиториев.
Используй только контекст ниже. Если информации недостаточно, скажи об этом прямо.

Контекст:
{context_text}
"""

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": question},
            ],
            temperature=0.1,
        )

        return response.choices[0].message.content, list(dict.fromkeys(sources))

## 11. Финальный запуск RAG

In [ ]:
# Для Colab можно загрузить ключ через userdata или задать вручную
# from google.colab import userdata
# api_key = userdata.get("POLZA_KEY")

api_key = os.getenv("POLZA_KEY", "")
base_url = "https://polza.ai/api/v1"

reranker = BGEReranker()

rag = RAGSystem(
    vector_store=vstore,
    api_key=api_key,
    base_url=base_url,
    reranker=reranker,
)

question = "Можешь в целом рассказать про что проект? Пришли список тестируемых моделей?"
answer_rag, sources = rag.ask(question)

print(f"ВОПРОС: {question}")
print(f"\nОТВЕТ RAG:\n{answer_rag}")
print(f"\nИСТОЧНИКИ:\n" + "\n".join(sources))

## 12. Сверка контекста: без реранкера и с реранкером

Ниже — отдельная проверка того, какие чанки приходят:
- сразу после поиска в векторной базе,
- и после дополнительного реранкинга.

In [ ]:
questions = [
    "Можешь в целом рассказать про что проект?",
    "Пришли список тестируемых моделей?",
    "Какие метрики оценки модели?",
    "Объясни механизм оценки модели?",
    "Какая модель получила наибольший скор?",
]

In [ ]:
results = []

for question in questions:
    raw_results = vstore.search(question, limit=10)
    reranked_results = reranker.rerank(question, raw_results, top_n=4)

    kept_chunks = set()
    for res in reranked_results:
        chunk_key = (
            res["payload"].get("file_path"),
            res["payload"].get("content"),
        )
        kept_chunks.add(chunk_key)

    question_result = {
        "question": question,
        "chunks": []
    }

    for idx, res in enumerate(raw_results, start=1):
        chunk_key = (
            res.payload.get("file_path"),
            res.payload.get("content"),
        )

        status = "оставлен" if chunk_key in kept_chunks else "отброшен reranker"

        question_result["chunks"].append({
            "chunk_number": idx,
            "status": status,
            "score_before_reranker": float(res.score),
            "file_path": res.payload.get("file_path"),
            "source_url": res.payload.get("source_url"),
            "content": res.payload.get("content"),
        })

    results.append(question_result)

print("Сравнение сохранено в results")

In [ ]:
for q_idx, item in enumerate(results, start=1):
    print("=" * 100)
    print(f"Вопрос {q_idx}: {item['question']}")
    print("=" * 100)

    for chunk in item["chunks"]:
        print(f"\nchunk {chunk['chunk_number']} - {chunk['status']}")
        print(f"file: {chunk['file_path']}")
        print(f"source: {chunk['source_url']}")
        print("контекст:")
        print(chunk["content"][:1000])
        print("-" * 100)

## 13. Формирование JSON-ответа и сохранение

Следующий код:
- берёт пользовательский вопрос,
- получает ответ по чанкам без реранкера,
- получает ответ по чанкам после реранкера,
- собирает всё в один JSON,
- и сохраняет его в файл.

In [ ]:
def build_context_from_results(search_results):
    context_text = ""
    chunks = []

    for res in search_results:
        if isinstance(res, dict):
            score = float(res["score"])
            payload = res["payload"]
        else:
            score = float(res.score)
            payload = res.payload

        content = payload["content"]
        path = payload["file_path"]
        url = payload.get("source_url", "No URL")

        context_text += f"\n---\nFILE: {path}\nCONTENT:\n{content}\n"
        chunks.append({
            "score": score,
            "file_path": path,
            "source_url": url,
            "content": content,
        })

    return context_text, chunks


def generate_answer_from_context(question: str, context_text: str, model_name: str, client):
    system_prompt = f"""Ты русский эксперт. Используй только предоставленный контекст для ответа на вопрос пользователя.
Если информации в контексте недостаточно, прямо скажи об этом.

Контекст:
{context_text}
"""

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        temperature=0.1,
    )

    return response.choices[0].message.content

In [ ]:
answers = []

for question in questions:
    search_without_reranker = vstore.search(question, limit=4)
    context_without_reranker, chunks_without_reranker = build_context_from_results(search_without_reranker)

    answer_without_reranker = generate_answer_from_context(
        question=question,
        context_text=context_without_reranker,
        model_name=rag.model,
        client=rag.client,
    )

    search_before_rerank = vstore.search(question, limit=10)
    search_with_reranker = reranker.rerank(question, search_before_rerank, top_n=4)
    context_with_reranker, chunks_with_reranker = build_context_from_results(search_with_reranker)

    answer_with_reranker = generate_answer_from_context(
        question=question,
        context_text=context_with_reranker,
        model_name=rag.model,
        client=rag.client,
    )

    answer_item = {
        "question": question,
        "answer_without_reranker": answer_without_reranker,
        "answer_with_reranker": answer_with_reranker,
        "query_with_chunks_without_reranker": context_without_reranker,
        "query_with_chunks_after_reranker": context_with_reranker,
        "chunks_without_reranker": chunks_without_reranker,
        "chunks_after_reranker": chunks_with_reranker,
        "meta": {
            "llm_model_name": rag.model,
            "embedding_model_name": vstore.embedding_model_name,
            "reranker_name": reranker.model_name,
            "vector_store": "Qdrant (:memory:)",
            "generated_at": datetime.now(UTC).isoformat(),
        }
    }

    answers.append(answer_item)

print("Ответы сохранены в answers")

In [ ]:
for i, item in enumerate(answers, start=1):
    print("=" * 100)
    print(f"Вопрос {i}: {item['question']}")
    print("=" * 100)

    print("\nОтвет без реранкера:")
    print(item["answer_without_reranker"])

    print("\nОтвет с реранкером:")
    print(item["answer_with_reranker"])

    print("\n" + "-" * 100 + "\n")

In [ ]:
output_json_path = "rag_answers_comparison.json"

with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(answers, f, ensure_ascii=False, indent=2)

print(f"JSON сохранён в файл: {output_json_path}")